In [ ]:
import os
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader,Dataset
from torchvision import datasets,transforms
from PIL import Image
import numpy as np
from tqdm import tqdm


In [ ]:
import torch
print(torch.device)
print(type(torch.device))
print(torch.cuda.is_available())

<class 'torch.device'>
<class 'type'>
True


In [ ]:
from torchvision.transforms.transforms import Normalize
train_transform=transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224,224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(0.3),
    transforms.ToTensor(),
    # transforms.Normalize([0.5], [0.5])
    transforms.Lambda(lambda x: torch.clamp(x, -1000, 400)/1400)

])
test_transform=transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224,224)),
    transforms.transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(0.3),
    transforms.transforms.ToTensor(),
    # transforms.Normalize([0.5],[0.5])
    transforms.Lambda(lambda x:torch.clamp(x,-1000,400)/1400)

])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.getcwd()

'/content'

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.block(x)
class Model_v2(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()

        # Backbone
        self.features = nn.Sequential(
            ConvBlock(in_channels, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            ConvBlock(256, 512)
        )

        # CAM Head
        self.cam_conv = nn.Conv2d(512, num_classes, kernel_size=1)

        # Classifier
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.features(x)           # [B, 512, H, W]
        heatmap = self.cam_conv(x)     # [B, C, H, W]
        logits = self.pool(heatmap)    # [B, C, 1, 1]
        logits = logits.view(x.size(0), -1)

        return logits, heatmap

In [ ]:
from tqdm import tqdm

def train_test(model, num_epochs,train_data_loader,test_data_loader,patience=20):

    best_loss = float("inf")
    counter = 0

    for epoch in range(num_epochs):

        # =======================
        # Training
        # =======================
        model.train()
        train_loss, train_acc = 0.0, 0.0

        loop = tqdm(
            train_data_loader,
            desc=f"Epoch [{epoch+1}/{num_epochs}] Training",
            leave=False
        )

        for x, y in loop:
            x,y=x.to(device),y.to(device)
            y_pred, _ = model(x)

            loss = loss_fn(y_pred, y)
            pred = torch.argmax(y_pred, dim=1)
            acc = (pred == y).float().mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_acc += acc.item()

            loop.set_postfix(loss=loss.item(), acc=acc.item())

        train_loss /= len(train_data_loader)
        train_acc /= len(train_data_loader)

        # =======================
        # Testing
        # =======================
        model.eval()
        test_loss, test_acc = 0.0, 0.0

        with torch.inference_mode():
            for x_test, y_test in test_data_loader:
                x_test,y_test=x_test.to(device),y_test.to(device)
                y_pred_test, _ = model(x_test)

                loss_test = loss_fn(y_pred_test, y_test)
                pred_test = torch.argmax(y_pred_test, dim=1)
                acc = (pred_test == y_test).float().mean()

                test_loss += loss_test.item()
                test_acc += acc.item()

        test_loss /= len(test_data_loader)
        test_acc /= len(test_data_loader)

        # =======================
        # Early Stopping Logic
        # =======================
        if test_loss < best_loss:
            best_loss = test_loss
            counter = 0
            best_model_state = model.state_dict()
        else:
            counter += 1

        print(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} || "
            f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} "
            f"| ES Counter: {counter}/{patience}"
        )

        if counter >= patience:
            print("Early stopping triggered.")
            model.load_state_dict(best_model_state)
            break



In [ ]:
train_transform_mri=transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224,224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(0.3),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x - x.mean()) / (x.std() + 1e-5))]
)

test_transform_mri=transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.transforms.Lambda(lambda x:(x-x.mean()) / (x.std()+ 1e-5))
])




In [ ]:
train_transform_mri_tumor=train_transform_mri
test_transform_mri_tumor=train_transform_mri
train_transform_tumor=train_transform
test_transform_tumor=test_transform
device="cuda"


In [ ]:
full_data_path_tumor="/content/drive/MyDrive/data_tyumor/Dataset/Brain Tumor CT scan Images"

In [ ]:
# all_dataset_transform_tumor=datasets.ImageFolder(
#     root=full_data_path_tumor,
#     transform=None,
#     target_transform=None
# )

# train_dataset_tumor,test_dataset_tumor=torch.utils.data.random_split(all_dataset_transform_tumor,[train_size,test_size])
# # print(len(all_dataset_transform_tumor.classes))
# train_dataset_tumor.transform =train_transform_tumor
# test_dataset_tumor.transform=test_transform_tumor


all_dataset_transform_tumor = datasets.ImageFolder(
    root=full_data_path_tumor,
    transform=train_transform_tumor,  # هذا الآن للـ train
    target_transform=None
)
train_size=int(0.8*len(all_dataset_transform_tumor))
test_size=len(all_dataset_transform_tumor)-train_size
train_dataset_tumor, test_dataset_tumor = torch.utils.data.random_split(all_dataset_transform_tumor, [train_size, test_size])

test_dataset_tumor.dataset.transform = test_transform

In [ ]:
train_dataset_tumor_loader=DataLoader(
    dataset=train_dataset_tumor,#don't use train_dataset_tumor.dataset->it wll return to the main_datasets
    batch_size=32,
    shuffle=True
)
test_dataset_tumor_loader=DataLoader(
    dataset=test_dataset_tumor,
    batch_size=32,
    shuffle=True
)

In [ ]:
device="cuda"

In [ ]:
modelv2_tumor=Model_v2(in_channels=1).to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=modelv2_tumor.parameters(),lr=0.03)
train_test(modelv2_tumor,num_epochs=60,patience=15,train_data_loader=train_dataset_tumor_loader,test_data_loader=test_dataset_tumor_loader)

Epoch [1/60] Train Loss: 0.5837 | Train Acc: 0.8367 || Test Loss: 2.3150 | Test Acc: 0.5802 | ES Counter: 0/15


Epoch [2/60] Train Loss: 0.2564 | Train Acc: 0.9120 || Test Loss: 0.2549 | Test Acc: 0.9039 | ES Counter: 0/15


Epoch [3/60] Train Loss: 0.3018 | Train Acc: 0.8869 || Test Loss: 0.1894 | Test Acc: 0.9363 | ES Counter: 0/15


Epoch [4/60] Train Loss: 0.2013 | Train Acc: 0.9275 || Test Loss: 0.2217 | Test Acc: 0.9293 | ES Counter: 1/15


Epoch [5/60] Train Loss: 0.1766 | Train Acc: 0.9316 || Test Loss: 0.1944 | Test Acc: 0.9341 | ES Counter: 2/15


...

Epoch [56/60] Train Loss: 0.0339 | Train Acc: 0.9897 || Test Loss: 0.0786 | Test Acc: 0.9731 | ES Counter: 2/15


Epoch [57/60] Train Loss: 0.0292 | Train Acc: 0.9895 || Test Loss: 0.0826 | Test Acc: 0.9697 | ES Counter: 3/15


Epoch [58/60] Train Loss: 0.0253 | Train Acc: 0.9916 || Test Loss: 0.0526 | Test Acc: 0.9881 | ES Counter: 4/15


Epoch [59/60] Train Loss: 0.0220 | Train Acc: 0.9925 || Test Loss: 0.0635 | Test Acc: 0.9784 | ES Counter: 5/15


Epoch [60/60] Train Loss: 0.0261 | Train Acc: 0.9919 || Test Loss: 0.4059 | Test Acc: 0.8842 | ES Counter: 6/15


In [ ]:
loaded_model_tumor=Model_v2(in_channels=1)
torch.save(modelv2_tumor.state_dict(),"model_tumor_weights.pth")
loaded_model_tumor.load_state_dict(torch.load("model_tumor_weights.pth"))
torch.save(modelv2_tumor.features.state_dict(),"encoder_tumor_ct.pth")
encoder_tumor_ct=torch.load("encoder_tumor_ct.pth")

In [ ]:
full_data_path_tumor_mri="/content/drive/MyDrive/data_tyumor/Dataset/Brain Tumor MRI images"
all_dataset_transform_tumor_mri=datasets.ImageFolder(
    root=full_data_path_tumor_mri,
    transform=train_transform_mri_tumor,
    target_transform=None
)
train_size=int(0.8*len(all_dataset_transform_tumor_mri))
test_size=len(all_dataset_transform_tumor_mri)-train_size
train_dataset_tumor_mri,test_dataset_tumor_mri=torch.utils.data.random_split(all_dataset_transform_tumor_mri,[train_size,test_size])
train_dataset_tumor_mri.dataset.classes
# train_dataset_tumor_mri.transform=train_transform_mri_tumor
test_dataset_tumor_mri.transform=test_transform_mri_tumor



In [ ]:
print(f"the length of full data={len(all_dataset_transform_tumor_mri)}")
print(f"the length of training data={len(train_dataset_tumor_mri)} and the length of test data={len(test_dataset_tumor_mri)}")

the length of full data=3634
the length of training data=2907 and the length of test data=727


In [ ]:
train_dataset_tumor_mri_loader=DataLoader(
    dataset=train_dataset_tumor,#don't use train_dataset_tumor.dataset->it wll return to the main_datasets
    batch_size=32,
    shuffle=True
)
test_dataset_tumor_mri_loader=DataLoader(
    dataset=test_dataset_tumor,
    batch_size=32,
    shuffle=True
)

In [ ]:
model_v2_tumor_mri=Model_v2(in_channels=1).to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=model_v2_tumor_mri.parameters(),lr=0.03)
train_test(model_v2_tumor_mri,num_epochs=60,train_data_loader=train_dataset_tumor_mri_loader,test_data_loader=test_dataset_tumor_loader)


Epoch [1/60] Train Loss: 0.5769 | Train Acc: 0.8225 || Test Loss: 15.8106 | Test Acc: 0.4948 | ES Counter: 0/20


Epoch [2/60] Train Loss: 0.2714 | Train Acc: 0.9099 || Test Loss: 0.6251 | Test Acc: 0.7118 | ES Counter: 0/20


Epoch [3/60] Train Loss: 0.2028 | Train Acc: 0.9275 || Test Loss: 0.6168 | Test Acc: 0.5574 | ES Counter: 0/20


Epoch [4/60] Train Loss: 0.1908 | Train Acc: 0.9324 || Test Loss: 0.2797 | Test Acc: 0.9164 | ES Counter: 0/20


Epoch [5/60] Train Loss: 0.1710 | Train Acc: 0.9410 || Test Loss: 0.2115 | Test Acc: 0.9153 | ES Counter: 0/20


...

Epoch [56/60] Train Loss: 0.0309 | Train Acc: 0.9903 || Test Loss: 0.1053 | Test Acc: 0.9607 | ES Counter: 13/20


Epoch [57/60] Train Loss: 0.0344 | Train Acc: 0.9878 || Test Loss: 0.0462 | Test Acc: 0.9892 | ES Counter: 0/20


Epoch [58/60] Train Loss: 0.0456 | Train Acc: 0.9833 || Test Loss: 0.0743 | Test Acc: 0.9784 | ES Counter: 1/20


Epoch [59/60] Train Loss: 0.0305 | Train Acc: 0.9892 || Test Loss: 0.1100 | Test Acc: 0.9649 | ES Counter: 2/20


Epoch [60/60] Train Loss: 0.0400 | Train Acc: 0.9863 || Test Loss: 0.1290 | Test Acc: 0.9535 | ES Counter: 3/20


In [ ]:
torch.save(model_v2_tumor_mri.state_dict(),"model_weights_tumor_mri_second.pth")
torch.save(model_v2_tumor_mri.features.state_dict(),"encoder_tumor_mri_second.pth")